# Lab 1：静态页面与 API 数据采集

目标：采集 SmartPark DataLab 的全部 Device 数据，以及连续 3 页的 Company 数据；再完成数量、唯一性和跨数据源一致性验证。

In [1]:
from collections import Counter
from pathlib import Path
from urllib.parse import urljoin
import json

import requests
from bs4 import BeautifulSoup

BASE_URL = 'https://115.159.159.82'
API_URL = f'{BASE_URL}/api/devices'
COMPANIES_URL = f'{BASE_URL}/companies'
TIMEOUT = 15

session = requests.Session()
session.headers.update({'User-Agent': 'Mozilla/5.0 (Lab1 data collection)'})

## 任务 1 与 2：Device 动态加载与 API 分页采集

通过浏览器 Network 的 Fetch/XHR 可定位到 `GET /api/devices`。接口使用 `page` 和 `page_size` 两个查询参数，响应的分页字段为 `page`、`page_size`、`total`、`items`。

In [2]:
first_response = session.get(API_URL, params={'page': 1, 'page_size': 50}, timeout=TIMEOUT)
first_response.raise_for_status()
first_page = first_response.json()

total = first_page['total']
page_size = first_page['page_size']
total_pages = (total + page_size - 1) // page_size

print('status:', first_response.status_code)
print('url:', first_response.url)
print('content-type:', first_response.headers.get('Content-Type'))
print('JSON 字段:', list(first_page.keys()))
print('每页条数:', page_size)
print('设备总数:', total)
print('计算总页数:', total_pages)
print('第一条设备:', first_page['items'][0])

status: 200
url: https://115.159.159.82/api/devices?page=1&page_size=50
content-type: application/json
JSON 字段: ['page', 'page_size', 'total', 'items']
每页条数: 50
设备总数: 5000
计算总页数: 100
第一条设备: {'device_id': 'D00000001', 'company_id': 'C000001', 'device_name': '循环泵-000001-001', 'device_type': 'pump', 'model': 'SP-PUM-800', 'zone': '智造一区', 'install_date': '2021-09-19', 'rated_power_kw': 71.2, 'sampling_interval_s': 30, 'protocol': 'mqtt', 'status': 'running', 'updated_at': '2026-09-01T03:58:00'}


In [3]:
devices = []

for page in range(1, total_pages + 1):
    response = session.get(
        API_URL,
        params={'page': page, 'page_size': page_size},
        timeout=TIMEOUT,
    )
    response.raise_for_status()
    page_data = response.json()
    devices.extend(page_data['items'])

print('最终采集设备数:', len(devices))
print('唯一 device_id 数:', len({item['device_id'] for item in devices}))

最终采集设备数: 5000
唯一 device_id 数: 5000


## 任务 3 与 4：Company 静态 HTML 与“下一页”分页

企业卡片位于静态 HTML 中。使用 `article.company-card` 定位一条完整记录，再从记录内部提取文本和 `data-company-id` 属性。使用 `nav.pagination a` 定位下一页链接，并用 `urljoin` 处理相对 URL。

In [4]:
def parse_company_card(card):
    details = {}
    for item in card.select('dl > div'):
        key = item.select_one('dt').get_text(strip=True)
        value = item.select_one('dd').get_text(strip=True)
        details[key] = value

    return {
        'company_id': card['data-company-id'],
        'name': card.select_one('h2 a').get_text(strip=True),
        'industry': details['所属行业'],
        'zone': details['所在区域'],
        'device_count': int(details['设备数量'].replace(' 台', '')),
        'status': card.select_one('.company-card-top .badge').get_text(strip=True),
    }

companies = []
page_counts = []
current_url = COMPANIES_URL

for page_number in range(1, 4):
    response = session.get(current_url, timeout=TIMEOUT)
    response.raise_for_status()
    soup = BeautifulSoup(response.text, 'html.parser')
    cards = soup.select('article.company-card')
    page_companies = [parse_company_card(card) for card in cards]
    companies.extend(page_companies)
    page_counts.append(len(page_companies))

    next_link = soup.select_one('nav.pagination a:last-of-type')
    if page_number < 3:
        if next_link is None:
            raise RuntimeError('第 {} 页未找到下一页链接'.format(page_number))
        current_url = urljoin(current_url, next_link['href'])

print('三页企业数:', page_counts)
print('Company 总数:', len(companies))
print('第 1 页下一页 href:', '/companies?page=2')
print('第一家企业:', companies[0])

三页企业数: [20, 20, 20]
Company 总数: 60
第 1 页下一页 href: /companies?page=2
第一家企业: {'company_id': 'C000001', 'name': '临港精工001有限公司', 'industry': '汽车零部件', 'zone': '智造一区', 'device_count': 6, 'status': '运营中'}


## 任务 5：结果验证

In [5]:
device_counts = Counter(item['company_id'] for item in devices)
mismatches = []

for company in companies:
    company_id = company['company_id']
    expected = company['device_count']
    actual = device_counts[company_id]
    if expected != actual:
        mismatches.append({
            'company_id': company_id,
            'expected': expected,
            'actual': actual,
        })

print('Company 总数:', len(companies))
print('唯一 company_id 数:', len({item['company_id'] for item in companies}))
print('Device 总数:', len(devices))
print('唯一 device_id 数:', len({item['device_id'] for item in devices}))
print('Company–Device 不一致企业数:', len(mismatches))
assert len(companies) == 60
assert len({item['company_id'] for item in companies}) == 60
assert len(devices) == 5000
assert len({item['device_id'] for item in devices}) == 5000
assert not mismatches

Company 总数: 60
唯一 company_id 数: 60
Device 总数: 5000
唯一 device_id 数: 5000
Company–Device 不一致企业数: 0


## 任务 6：保存 JSON 文件

In [6]:
data_dir = Path('data')
data_dir.mkdir(exist_ok=True)

with open(data_dir / 'devices.json', 'w', encoding='utf-8') as file:
    json.dump(devices, file, ensure_ascii=False, indent=2)

with open(data_dir / 'companies.json', 'w', encoding='utf-8') as file:
    json.dump(companies, file, ensure_ascii=False, indent=2)

print('已生成:', data_dir / 'devices.json')
print('已生成:', data_dir / 'companies.json')

已生成: data/devices.json
已生成: data/companies.json
